In [16]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import BaggingClassifier, VotingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import time


In [2]:

# load dataset
data = pd.read_csv('../data/rt_iot2022.csv')

In [3]:
data

,id.orig_p,id.resp_p,proto,service,flow_duration,fwd_pkts_tot,bwd_pkts_tot,fwd_data_pkts_tot,bwd_data_pkts_tot,fwd_pkts_per_sec,...,active.std,idle.min,idle.max,idle.tot,idle.avg,idle.std,fwd_init_window_size,bwd_init_window_size,fwd_last_window_size,Attack_type
0,38667,1883,tcp,mqtt,32.011598,9,5,3,3,0.281148,...,0.0,29729182.96,29729182.96,29729182.96,29729182.96,0.0,64240,26847,502,MQTT_Publish
1,51143,1883,tcp,mqtt,31.883584,9,5,3,3,0.282277,...,0.0,29855277.06,29855277.06,29855277.06,29855277.06,0.0,64240,26847,502,MQTT_Publish
2,44761,1883,tcp,mqtt,32.124053,9,5,3,3,0.280164,...,0.0,29842149.02,29842149.02,29842149.02,29842149.02,0.0,64240,26847,502,MQTT_Publish
3,60893,1883,tcp,mqtt,31.961063,9,5,3,3,0.281593,...,0.0,29913774.97,29913774.97,29913774.97,29913774.97,0.0,64240,26847,502,MQTT_Publish
4,51087,1883,tcp,mqtt,31.902362,9,5,3,3,0.282111,...,0.0,29814704.90,29814704.90,29814704.90,29814704.90,0.0,64240,26847,502,MQTT_Publish
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123112,59247,63331,tcp,-,0.000006,1,1,0,0,167772.160000,...,0.0,0.00,0.00,0.00,0.00,0.0,1024,0,1024,NMAP_XMAS_TREE_SCAN
123113,59247,64623,tcp,-,0.000007,1,1,0,0,144631.172400,...,0.0,0.00,0.00,0.00,0.00,0.0,1024,0,1024,NMAP_XMAS_TREE_SCAN
123114,59247,64680,tcp,-,0.000006,1,1,0,0,167772.160000,...,0.0,0.00,0.00,0.00,0.00,0.0,1024,0,1024,NMAP_XMAS_TREE_SCAN
123115,59247,65000,tcp,-,0.000006,1,1,0,0,167772.160000,...,0.0,0.00,0.00,0.00,0.00,0.0,1024,0,1024,NMAP_XMAS_TREE_SCAN


In [4]:
X = data.drop(columns=['Attack_type'])
y = data['Attack_type']

In [7]:
X = X.select_dtypes(include=[np.number])

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training Shape: {X_train.shape}")
print(f"Testing Shape: {X_test.shape}")

Training Shape: (98493, 81)
Testing Shape: (24624, 81)


Decision Tree Optimization

In [13]:
dt_param_grid = {
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy']
}

In [15]:
dt_grid = GridSearchCV(DecisionTreeClassifier(random_state=42), dt_param_grid, cv=5, scoring='accuracy')
dt_grid.fit(X_train, y_train)
best_dt = dt_grid.best_estimator_
print(f"Best DT Params: {dt_grid.best_params_}")
print(f"Best DT CV Score: {dt_grid.best_score_}")

Best DT Params: {'criterion': 'entropy', 'max_depth': None, 'min_samples_split': 2}


SVM Optimization

In [17]:
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(probability=True, random_state=42))
])

In [18]:
svm_param_grid = {
    'svm__C': [0.1, 1, 10],
    'svm__kernel': ['linear', 'rbf'],
    'svm__gamma': ['scale', 'auto']
}

In [ ]:
svm_grid = GridSearchCV(svm_pipeline, svm_param_grid, cv=5, scoring='accuracy')
svm_grid.fit(X_train, y_train)
best_svm_pipeline = svm_grid.best_estimator_
print(f"Best SVM Params: {svm_grid.best_params_}")
print(f"Best SVM CV Score: {svm_grid.best_score_}")

Ensemble Methods

Bagging Decision Tree

In [ ]:
bagging_dt = BaggingClassifier(
    estimator=best_dt,
    n_estimators=50,
    max_samples=0.8,
    random_state=42
)

In [ ]:
bagging_dt.fit(X_train, y_train)

Bagging SVM

In [ ]:
bagging_svm = BaggingClassifier(
    estimator=best_svm_pipeline,
    n_estimators=10,     # Fewer estimators for SVM as it is slower
    max_samples=0.8,
    random_state=42
)


In [ ]:
bagging_svm.fit(X_train, y_train)

Voting Classifier

In [ ]:
voting_clf = VotingClassifier(
    estimators=[
        ('bagging_dt', bagging_dt),
        ('bagging_svm', bagging_svm)
    ],
    voting='soft' # Soft voting uses predicted probabilities [cite: 98]
)
voting_clf.fit(X_train, y_train)

In [ ]:
models = {
    "Optimized DT": best_dt,
    "Optimized SVM": best_svm_pipeline,
    "Bagging DT": bagging_dt,
    "Bagging SVM": bagging_svm,
    "Voting Soft": voting_clf
}

results = []

for name, model in models.items():
    start = time.time()
    y_pred = model.predict(X_test)
    inference_time = (time.time() - start) * 1000 # ms

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted') # Weighted for imbalance handling

    results.append({
        "Model": name,
        "Accuracy": acc,
        "F1-Score": f1,
        "Inference Time (ms)": inference_time
    })

    print(f"-> {name} processed.")

# Create Comparison Table
results_df = pd.DataFrame(results).sort_values(by="Accuracy", ascending=False)
print("\n", results_df)